# 第一段階：角度較正前後の誤差チャネル比較

有限 Lamb–Dicke MS ゲートについて、矩形パルスの振幅を \(h_{XX}=0\) に近づけるよう較正し、その前後で**誤差チャネル全体**を比較する。

中心となる問いは次の三つである。

1. 角度較正により Hamiltonian 型の \(h_{XX}\) は実際に除去されるか。
2. 較正後も \(\gamma_{XX}\)、\(\gamma_{\rm col}\)、平均ゲート infidelity は残るか。
3. 三成分生成子
   
   \[
   K_3=h_{XX}\mathcal H[XX]+\gamma_{XX}\mathcal D[XX]
       +\gamma_{\rm col}\mathcal D[IX+XI]
   \]
   
   は較正前後の full QPT 生成子をどこまで説明できるか。

このノートの較正は各 \(\bar n\) で独立に行うため、角度誤差を最もよく除いた best-case 比較である。一つの温度で得た較正値を別の温度へ移す検証は次段階とする。

## 0. 判定の考え方

- \(h_{XX}\) が大きく減る一方で infidelity があまり減らなければ、単一角度較正では除去できない誤差床の候補となる。
- \(\gamma_{XX}\) と \(\gamma_{\rm col}\) は較正後に必ず不変とは限らない。振幅変更後の QPT から毎回取り直す。
- \(\mathcal D[IX+XI]\) なら完全な H/S/C/A 分解で \(S_{IX}=S_{XI}=C_{IX,XI}\) が期待される。その係数の spread を collective-X 仮定の診断に使う。
- 三成分残差が大きければ、三成分は説明用の低次元近似にすぎず、チャネルの普遍的・厳密な記述とはみなさない。

以下の閾値は普遍的な物理定数ではなく、この解析で仮説を棄却しやすくするための編集可能な operational criterion である。

In [16]:
from pathlib import Path
import importlib
import json
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import scipy.linalg
import scipy.optimize
from IPython.display import Markdown, display


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "chi_error_nbar_workflow.py").exists():
            return candidate
    raise FileNotFoundError("chi_error_nbar_workflow.py がある project root を見つけられません")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import chi_error_nbar_workflow as workflow
import chi_error_nbar_stages as stages
import drive_calibration_qpt_analysis as qpt_analysis
import ms_gate_functions as mg
import noise_error_structure as noise_structure

workflow = importlib.reload(workflow)
stages = importlib.reload(stages)
qpt_analysis = importlib.reload(qpt_analysis)
noise_structure = importlib.reload(noise_structure)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:.6g}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /workspace


## 1. 設定

既定では QPT を再計算せず、既存キャッシュだけを読む。不足キャッシュを計算するときだけ RUN_QPT を True にする。FORCE_RECOMPUTE は既存キャッシュまで上書きするため、通常は False のままにする。

In [17]:
CONFIG = workflow.default_config()
CONFIG["OUTPUT_DIR"] = str(PROJECT_ROOT / "results" / "chi_error_element_fit")


WORKER_COUNT = 24
if WORKER_COUNT < 1:
    raise ValueError("WORKER_COUNT must be at least 1")
CONFIG["PARALLEL_WORKERS"] = WORKER_COUNT
CONFIG["FAST_PROCESS_WORKERS"] = WORKER_COUNT
CONFIG["SIMULATION_PARAMS"]["parallel_workers"] = WORKER_COUNT

NBAR_VALUES = [0.01, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0]
RUN_QPT = True
FORCE_RECOMPUTE = False
MAX_FEEDBACK_ITERATIONS = 2
SELECTED_NBAR_FOR_CHI = 4.0
CHI_MATRIX_NBARS = [SELECTED_NBAR_FOR_CHI]  # 複素 chi 行列を直接表示する温度

HXX_TOL_RAD_PER_GATE = 2e-3
MODEL_RESIDUAL_FRACTION_MAX = 0.25
COLLECTIVE_SPREAD_RELATIVE_MAX = 0.25
STRONG_HXX_REDUCTION_FACTOR = 10.0
INFIDELITY_FLOOR_REMAINING_MIN = 0.50

ANALYSIS_DIR = PROJECT_ROOT / "results" / "calibration_channel_comparison"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

workflow.validate_config(CONFIG)
display(pd.Series({
    "worker_count": WORKER_COUNT,
    "nbar_values": NBAR_VALUES,
    "run_qpt": RUN_QPT,
    "force_recompute": FORCE_RECOMPUTE,
    "max_feedback_iterations": MAX_FEEDBACK_ITERATIONS,
    "analysis_dir": str(ANALYSIS_DIR),
}, name="value").to_frame())

,value
worker_count,24
nbar_values,"[0.01, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0]"
run_qpt,True
force_recompute,False
max_feedback_iterations,2
analysis_dir,/workspace/results/calibration_channel_comparison


## 2. 振幅較正と full-Hamiltonian QPT

較正前の \(h_{XX}\) から

\[
A_{k+1}=A_k\sqrt{\frac{\pi/4}{\pi/4-h_{XX}^{(k)}}}
\]

を予測値として用い、その振幅で full-Hamiltonian QPT をやり直す。高次補正により応答は厳密な二次則ではないため、残留 \(h_{XX}\) が閾値を超えた場合だけフィードバックを繰り返す。

In [3]:
drive_result = stages.run_drive_feedback_stage(
    CONFIG,
    NBAR_VALUES,
    run_qpt=RUN_QPT,
    force_recompute=FORCE_RECOMPUTE,
    max_feedback_iterations=MAX_FEEDBACK_ITERATIONS,
)

print(json.dumps(drive_result["status"], indent=2, ensure_ascii=False, default=str))
if drive_result["status"].get("pending"):
    display(pd.DataFrame(drive_result["status"]["pending"]))

drive_summary = drive_result["summary"].copy()
if drive_summary.empty:
    raise RuntimeError(
        "較正後 QPT がありません。不足点を計算する場合は RUN_QPT=True にしてこのセルを再実行してください。"
    )

display(drive_summary[[
    "n_bar", "iteration", "A_factor",
    "h_XX_before_rad_per_gate", "h_XX_after_rad_per_gate",
    "average_infidelity_before", "average_infidelity_after",
    "h_XX_converged",
]])

{
  "requested_nbars": [
    0.01,
    1.0,
    2.0,
    4.0,
    6.0,
    8.0,
    10.0
  ],
  "completed_nbars": [
    0.01,
    1.0,
    2.0,
    4.0,
    6.0,
    8.0,
    10.0
  ],
  "completed_count": 7,
  "expected_count": 7,
  "pending": [],
  "run_qpt": true,
  "publication_completion": {
    "completed_count": 10,
    "expected_count": 10,
    "converged_count": 10,
    "checklist_row": {
      "check": "hXX-derived drive calibration re-QPT",
      "status": "complete",
      "result": "10/10 temperatures re-QPT; 10 with |h_XX| <= 2.0e-03 rad/gate"
    },
    "checklist_path": "/Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/chi_error_element_fit/advanced_publication_validation/advanced_publication_checklist.csv",
    "manifest_path": "/Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/chi_error_element_fit/advanced_publication_validation/advanced_publication_manifest.json"
  }
}


,n_bar,iteration,A_factor,h_XX_before_rad_per_gate,h_XX_after_rad_per_gate,average_infidelity_before,average_infidelity_after,h_XX_converged
0,0.01,1,1.00886,0.0137365,0.000102408,0.00194653,0.00181869,True
1,1,1,1.01886,0.0288062,0.000212392,0.00364438,0.00310427,True
2,2,1,1.02907,0.0437398,0.000312228,0.00598075,0.00479633,True
3,4,1,1.04977,0.0727108,0.00043429,0.0122808,0.0093656,True
4,6,1,1.07083,0.100464,0.000363092,0.0203734,0.015512,True
5,8,1,1.09221,0.127023,-2.82896e-05,0.0298297,0.0231894,True
6,10,1,1.11393,0.152443,-0.000872615,0.0402815,0.0323355,True
7,12,2,1.13434,0.176799,3.49149e-05,0.0514219,0.04267,True
8,16,2,1.17579,0.222628,0.000289356,0.0748138,0.0668276,True
9,20,2,1.21591,0.265083,0.00120335,0.09853,0.0944349,True


## 3. チャネルから三成分生成子を抽出

In [4]:
def legacy_nbar_stem(n_bar):
    return str(float(n_bar)).replace("-", "m").replace(".", "p")


def baseline_chi_path(n_bar):
    return (
        Path(CONFIG["OUTPUT_DIR"])
        / "exact_full_chi"
        / f"error_chi_nbar_{legacy_nbar_stem(n_bar)}.npz"
    )


def resolve_project_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_trace_normalized_chi(path_like):
    path = resolve_project_path(path_like)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        chi = np.asarray(data["chi_trace_normalized"], dtype=complex)
    return chi, path


def build_three_component_bases():
    labels, hamiltonian_bases, _, dissipator_design = qpt_analysis._generator_design_data()
    pauli_labels = list(labels[1:])
    pauli_qobjs = tuple(mg.two_qubit_pauli_basis())
    pauli_by_label = dict(zip(labels, pauli_qobjs))
    h_xx = np.asarray(hamiltonian_bases["XX"], dtype=float)
    d_xx = dissipator_design[:, pauli_labels.index("XX")].reshape(h_xx.shape)
    jump = pauli_by_label["IX"] + pauli_by_label["XI"]
    jump_squared = jump * jump
    d_col = qpt_analysis._action_to_ptm(
        lambda rho: jump * rho * jump - 0.5 * (jump_squared * rho + rho * jump_squared),
        pauli_qobjs,
    )
    return {"H_XX": h_xx, "D_XX": d_xx, "D_col": d_col}


THREE_COMPONENT_BASES = build_three_component_bases()


def taxonomy_lookup(taxonomy):
    return {
        f"{sector}:{mode}": float(value)
        for (sector, mode), value in zip(taxonomy["metadata"], taxonomy["coefficients"])
    }


def analyze_channel(chi):
    superoperator, projected_chi, projection_status = (
        qpt_analysis.project_trace_normalized_chi_to_cptp(
            chi,
            tolerance=CONFIG.get("CPTP_TOLERANCE", 1e-11),
            max_iterations=CONFIG.get("CPTP_MAX_ITERATIONS", 5000),
        )
    )
    ptm = np.asarray(mg.superoperator_to_ptm(superoperator), dtype=complex)
    generator_complex = scipy.linalg.logm(ptm)
    generator = np.real(generator_complex)

    h_basis = THREE_COMPONENT_BASES["H_XX"]
    d_xx_basis = THREE_COMPONENT_BASES["D_XX"]
    d_col_basis = THREE_COMPONENT_BASES["D_col"]

    skew = 0.5 * (generator - generator.T)
    h_xx = float(np.linalg.lstsq(h_basis.reshape(-1, 1), skew.reshape(-1), rcond=None)[0][0])
    h_fit = h_xx * h_basis
    symmetric_remaining = 0.5 * ((generator - h_fit) + (generator - h_fit).T)
    dissipator_design = np.column_stack([d_xx_basis.reshape(-1), d_col_basis.reshape(-1)])
    gamma, gamma_nnls_residual = scipy.optimize.nnls(
        dissipator_design, symmetric_remaining.reshape(-1)
    )
    gamma_xx, gamma_col = map(float, gamma)
    model_generator = h_fit + gamma_xx * d_xx_basis + gamma_col * d_col_basis

    taxonomy = noise_structure._decompose_generator_taxonomy(generator)
    coefficients = taxonomy_lookup(taxonomy)
    collective_coefficients = np.array([
        coefficients["S:IX"],
        coefficients["S:XI"],
        coefficients["C:IX,XI"],
    ])
    collective_mean = float(np.mean(collective_coefficients))
    collective_spread = float(np.ptp(collective_coefficients))

    labels = list(qpt_analysis._generator_design_data()[0])
    ii_index = labels.index("II")
    xx_index = labels.index("XX")
    generator_norm = float(np.linalg.norm(generator))
    model_residual_norm = float(np.linalg.norm(generator - model_generator))
    ptm_real = np.real(ptm)

    scalars = {
        "h_XX_rad_per_gate": h_xx,
        "gamma_XX_per_gate": gamma_xx,
        "gamma_col_per_gate": gamma_col,
        "average_infidelity": 4.0 / 5.0 * (1.0 - float(np.real(projected_chi[ii_index, ii_index]))),
        "abs_chi_II_XX": float(abs(projected_chi[ii_index, xx_index])),
        "chi_XX_XX": float(np.real(projected_chi[xx_index, xx_index])),
        "generator_frobenius_norm": generator_norm,
        "three_component_residual_norm": model_residual_norm,
        "three_component_residual_fraction": model_residual_norm / max(generator_norm, 1e-15),
        "three_component_channel_error_fraction": float(
            np.linalg.norm(scipy.linalg.expm(model_generator) - ptm_real)
            / max(np.linalg.norm(ptm_real), 1e-15)
        ),
        "gamma_nnls_residual": float(gamma_nnls_residual),
        "taxonomy_H_XX": coefficients["H:XX"],
        "taxonomy_S_XX": coefficients["S:XX"],
        "taxonomy_S_IX": coefficients["S:IX"],
        "taxonomy_S_XI": coefficients["S:XI"],
        "taxonomy_C_IX_XI": coefficients["C:IX,XI"],
        "collective_x_coefficient_mean": collective_mean,
        "collective_x_coefficient_spread": collective_spread,
        "collective_x_relative_spread": collective_spread / max(abs(collective_mean), 1e-15),
        "taxonomy_reconstruction_residual_fraction": float(
            np.linalg.norm(taxonomy["residual"]) / max(generator_norm, 1e-15)
        ),
        "generator_imaginary_fraction": float(
            np.linalg.norm(np.imag(generator_complex)) / max(generator_norm, 1e-15)
        ),
        "cptp_min_choi_eigenvalue": float(projection_status["min_choi_eigenvalue"]),
        "cptp_tp_frobenius_error": float(projection_status["tp_frobenius_error"]),
    }
    return {
        "scalars": scalars,
        "chi": projected_chi,
        "ptm": ptm_real,
        "generator": generator,
        "model_generator": model_generator,
    }

## 4. 較正前後を同じ規約で再解析する

既存 summary の数値をそのまま横に並べるだけでなく、較正前後の \(\chi\) を同じ CPTP 射影・matrix-log・三成分 fit に通す。これにより比較時の解析規約を揃える。

In [5]:
records = []
channel_objects = {}
post_gate_chi_objects = {}

for n_bar in NBAR_VALUES:
    matched = drive_summary.loc[np.isclose(drive_summary["n_bar"].astype(float), n_bar)]
    if matched.empty:
        raise ValueError(f"n_bar={n_bar:g} の較正結果がありません")
    calibrated_row = matched.sort_values("iteration").iloc[-1]
    conditions = [
        ("before", baseline_chi_path(n_bar), 1.0, 0),
        (
            "after",
            calibrated_row["cache_path"],
            float(calibrated_row["A_factor"]),
            int(calibrated_row["iteration"]),
        ),
    ]

    for condition, source_path, amplitude_factor, iteration in conditions:
        chi, resolved_path = load_trace_normalized_chi(source_path)
        analyzed = analyze_channel(chi)
        post_gate_chi_objects[(float(n_bar), condition)] = chi
        channel_objects[(float(n_bar), condition)] = analyzed
        records.append({
            "n_bar": float(n_bar),
            "condition": condition,
            "A_factor": amplitude_factor,
            "feedback_iteration": iteration,
            "source_path": str(resolved_path),
            **analyzed["scalars"],
        })

comparison_long = pd.DataFrame(records).sort_values(["n_bar", "condition"]).reset_index(drop=True)
comparison_long_path = ANALYSIS_DIR / "channel_comparison_long.csv"
comparison_long.to_csv(comparison_long_path, index=False)

before = comparison_long.query("condition == 'before'").drop(columns="condition")
after = comparison_long.query("condition == 'after'").drop(columns="condition")
paired = before.merge(after, on="n_bar", suffixes=("_before", "_after"))

paired["abs_h_XX_reduction_factor"] = (
    np.abs(paired["h_XX_rad_per_gate_before"])
    / np.maximum(np.abs(paired["h_XX_rad_per_gate_after"]), 1e-15)
)
paired["infidelity_remaining_fraction"] = (
    paired["average_infidelity_after"]
    / np.maximum(paired["average_infidelity_before"], 1e-15)
)
paired["infidelity_reduction_percent"] = 100.0 * (1.0 - paired["infidelity_remaining_fraction"])
paired["gamma_XX_change"] = paired["gamma_XX_per_gate_after"] - paired["gamma_XX_per_gate_before"]
paired["gamma_col_change"] = paired["gamma_col_per_gate_after"] - paired["gamma_col_per_gate_before"]
paired["h_XX_converged"] = np.abs(paired["h_XX_rad_per_gate_after"]) <= HXX_TOL_RAD_PER_GATE

paired_path = ANALYSIS_DIR / "channel_comparison_paired.csv"
paired.to_csv(paired_path, index=False)

ordered_nbars = np.asarray(NBAR_VALUES, dtype=float)
chi_before_stack = np.stack([channel_objects[(float(n), "before")]["chi"] for n in ordered_nbars])
chi_after_stack = np.stack([channel_objects[(float(n), "after")]["chi"] for n in ordered_nbars])
np.savez_compressed(
    ANALYSIS_DIR / "projected_chi_before_after.npz",
    n_bar=ordered_nbars,
    chi_before=chi_before_stack,
    chi_after=chi_after_stack,
)
post_gate_chi_before_stack = np.stack([
    post_gate_chi_objects[(float(n), "before")] for n in ordered_nbars
])
post_gate_chi_after_stack = np.stack([
    post_gate_chi_objects[(float(n), "after")] for n in ordered_nbars
])
np.savez_compressed(
    ANALYSIS_DIR / "post_gate_noise_chi_before_after.npz",
    n_bar=ordered_nbars,
    convention="N_post = S_noisy @ inverse(S_ideal)",
    pauli_labels=np.asarray(qpt_analysis._generator_design_data()[0]),
    chi_before=post_gate_chi_before_stack,
    chi_after=post_gate_chi_after_stack,
)

display(paired[[
    "n_bar", "A_factor_after",
    "h_XX_rad_per_gate_before", "h_XX_rad_per_gate_after", "abs_h_XX_reduction_factor",
    "gamma_XX_per_gate_before", "gamma_XX_per_gate_after",
    "gamma_col_per_gate_before", "gamma_col_per_gate_after",
    "average_infidelity_before", "average_infidelity_after", "infidelity_reduction_percent",
    "three_component_residual_fraction_before", "three_component_residual_fraction_after",
]])
print(f"Saved: {comparison_long_path}")
print(f"Saved: {paired_path}")

,n_bar,A_factor_after,h_XX_rad_per_gate_before,h_XX_rad_per_gate_after,abs_h_XX_reduction_factor,gamma_XX_per_gate_before,gamma_XX_per_gate_after,gamma_col_per_gate_before,gamma_col_per_gate_after,average_infidelity_before,average_infidelity_after,infidelity_reduction_percent,three_component_residual_fraction_before,three_component_residual_fraction_after
0,0.01,1.00886,0.0137365,0.000102408,134.135,0.000545104,0.000550905,0.000661033,0.000672391,0.00194653,0.00181869,6.56746,0.0378786,0.276555
1,1,1.01886,0.0288062,0.000212392,135.628,0.00099713,0.0010442,0.00118085,0.00123318,0.00364438,0.00310427,14.8204,0.0178812,0.151767
2,2,1.02907,0.0437398,0.000312228,140.089,0.00186595,0.00204295,0.00167714,0.0017995,0.00598075,0.00479633,19.8039,0.0117722,0.0972455
3,4,1.04977,0.0727108,0.00043429,167.425,0.00467436,0.00557183,0.00258662,0.00292998,0.0122808,0.0093656,23.7378,0.00709229,0.0497417
4,6,1.07083,0.100464,0.000363092,276.69,0.00865539,0.0111756,0.00340049,0.00406432,0.0203734,0.015512,23.8612,0.00514314,0.0297719
5,8,1.09221,0.127023,-2.82896e-05,4490.07,0.0135344,0.0188756,0.00412818,0.00520378,0.0298297,0.0231894,22.2609,0.00407614,0.0196152
6,10,1.11393,0.152443,-0.000872615,174.697,0.0190796,0.028696,0.00477833,0.00635017,0.0402815,0.0323355,19.7262,0.00340401,0.0138058


Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/channel_comparison_long.csv
Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/channel_comparison_paired.csv


## 5. 角度較正前後の \(\chi\) 行列を直接見る

研究ノートと同じ定義

\[
\mathcal{N}_{\rm post}=\mathcal{S}_{\rm noisy}\mathcal{S}_{\rm ideal}^{-1}
\]

から直接得た、trace-normalized な post-gate noise \(\chi\) を表示する。CPTP 射影後の行列ではなく、QPT キャッシュに保存された行列そのものを使う。各条件について左から実部、虚部、絶対値を線形スケールで示す。

ヒートマップでは微小な誤差要素を見えるようにするため \(\chi_{II,II}\) だけを色スケールから除く。その値を含む完全な行列は数値表に残している。

In [6]:
if CONFIG.get("ERROR_CHANNEL_CONVENTION") != "undo_before_actual":
    raise ValueError(
        "研究ノートの post-gate convention には ERROR_CHANNEL_CONVENTION='undo_before_actual' が必要です"
    )

pauli_labels = list(qpt_analysis._generator_design_data()[0])
ii_index = pauli_labels.index("II")
available_nbars = np.asarray(sorted(comparison_long["n_bar"].unique()), dtype=float)

for requested_nbar in CHI_MATRIX_NBARS:
    selected_nbar = float(available_nbars[np.argmin(np.abs(available_nbars - requested_nbar))])
    display(Markdown(rf"### $\bar n={selected_nbar:g}$"))

    for condition, condition_title in [
        ("before", "Before angle calibration"),
        ("after", "After angle calibration"),
    ]:
        post_gate_chi = np.asarray(
            post_gate_chi_objects[(selected_nbar, condition)], dtype=complex
        )
        if not np.isclose(np.trace(post_gate_chi), 1.0, atol=1e-10):
            raise ValueError(f"Trace-normalized chi expected; trace={np.trace(post_gate_chi)}")

        chi_for_display = post_gate_chi.copy()
        identity_value = chi_for_display[ii_index, ii_index]
        chi_for_display[ii_index, ii_index] = 0.0

        figure, _ = mg.plot_chi_matrix(
            chi_for_display,
            title=(
                rf"{condition_title}: post-gate noise channel at $\bar n={selected_nbar:g}$"
                + "\n"
                + rf"$\mathcal{{N}}_{{post}}=\mathcal{{S}}_{{noisy}}\mathcal{{S}}_{{ideal}}^{{-1}}$, "
                + rf"display-only $\chi_{{II,II}}=0$ (original {identity_value.real:.8f})"
            ),
            components=("real", "imag", "abs"),
            figsize=(15.0, 5.0),
            show_title=True,
            show_component_titles=True,
            show_axis_labels=True,
            tick_labelsize=9,
            colorbar_tick_labelsize=9,
        )
        output_stem = (
            f"post_gate_noise_chi_{condition}_nbar_{legacy_nbar_stem(selected_nbar)}"
        )
        output_png = ANALYSIS_DIR / f"{output_stem}.png"
        output_pdf = ANALYSIS_DIR / f"{output_stem}.pdf"
        figure.savefig(output_png, dpi=300, bbox_inches="tight")
        figure.savefig(output_pdf, bbox_inches="tight")
        display(figure)
        plt.close(figure)
        print(f"Saved: {output_png}")

### $\bar n=4$

<Figure size 1500x500 with 6 Axes>

Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/post_gate_noise_chi_before_nbar_4p0.png


<Figure size 1500x500 with 6 Axes>

Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/post_gate_noise_chi_after_nbar_4p0.png


## 6. 主結果：較正で消える成分と残る成分

左上で角度較正の成立を、中央二枚で stochastic 成分の残留を、右上でチャネル全体の infidelity を見る。下段右は、三成分を使うこと自体の妥当性診断である。

In [7]:
figure, axes = plt.subplots(2, 3, figsize=(16.0, 9.0))

panels = [
    (axes[0, 0], "h_XX_rad_per_gate", r"$h_{XX}$ (rad/gate)", False),
    (axes[0, 1], "gamma_XX_per_gate", r"$\gamma_{XX}$ (1/gate)", True),
    (axes[0, 2], "gamma_col_per_gate", r"$\gamma_{\rm col}$ (1/gate)", True),
    (axes[1, 0], "average_infidelity", "Average infidelity", True),
    (axes[1, 1], "three_component_residual_fraction", r"$\|K-K_3\|_F/\|K\|_F$", False),
]

for axis, column, ylabel, log_scale in panels:
    for condition, label, marker in [("before", "before calibration", "o"), ("after", "after calibration", "s")]:
        subset = comparison_long.query("condition == @condition").sort_values("n_bar")
        values = subset[column].to_numpy(float)
        if log_scale:
            values = np.maximum(np.abs(values), 1e-16)
        axis.plot(subset["n_bar"], values, marker=marker, linewidth=2.0, label=label)
    if log_scale:
        axis.set_yscale("log")
    axis.set_xlabel(r"Mean phonon number $\bar n$")
    axis.set_ylabel(ylabel)
    axis.grid(True, which="both", alpha=0.28)

axes[0, 0].axhline(0.0, color="black", linewidth=0.8)
axes[0, 0].axhspan(-HXX_TOL_RAD_PER_GATE, HXX_TOL_RAD_PER_GATE, color="tab:green", alpha=0.10)
axes[0, 0].legend()

axes[1, 2].plot(paired["n_bar"], paired["abs_h_XX_reduction_factor"], "o-", label=r"$|h_{XX}|$ reduction")
axes[1, 2].plot(paired["n_bar"], 1.0 / np.maximum(paired["infidelity_remaining_fraction"], 1e-15), "s-", label="infidelity reduction")
axes[1, 2].set_yscale("log")
axes[1, 2].set_xlabel(r"Mean phonon number $\bar n$")
axes[1, 2].set_ylabel("Reduction factor")
axes[1, 2].grid(True, which="both", alpha=0.28)
axes[1, 2].legend()

figure.suptitle("Error channel before and after XX-angle calibration", y=1.01, fontsize=15)
figure.tight_layout()
main_figure_png = ANALYSIS_DIR / "channel_before_after_main.png"
main_figure_pdf = ANALYSIS_DIR / "channel_before_after_main.pdf"
figure.savefig(main_figure_png, dpi=300, bbox_inches="tight")
figure.savefig(main_figure_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {main_figure_png}")

<Figure size 1600x900 with 6 Axes>

Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/channel_before_after_main.png


## 7. collective-X 仮定と三成分近似の診断

restricted fit の \(\gamma_{\rm col}\) が得られても、それだけでは \(\mathcal D[IX+XI]\) が正しいとは言えない。完全分解の \(S_{IX},S_{XI},C_{IX,XI}\) の一致と、full generator に対する残差を同時に確認する。

In [8]:
collective_columns = [
    "n_bar", "condition", "gamma_col_per_gate",
    "taxonomy_S_IX", "taxonomy_S_XI", "taxonomy_C_IX_XI",
    "collective_x_relative_spread", "three_component_residual_fraction",
    "three_component_channel_error_fraction",
]
collective_diagnostic = comparison_long[collective_columns].copy()
collective_diagnostic_path = ANALYSIS_DIR / "three_component_model_diagnostics.csv"
collective_diagnostic.to_csv(collective_diagnostic_path, index=False)
display(collective_diagnostic)

figure, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
after_rows = comparison_long.query("condition == 'after'").sort_values("n_bar")
for column, label, marker in [
    ("taxonomy_S_IX", r"$S_{IX}$", "o"),
    ("taxonomy_S_XI", r"$S_{XI}$", "s"),
    ("taxonomy_C_IX_XI", r"$C_{IX,XI}$", "^"),
]:
    axes[0].plot(after_rows["n_bar"], after_rows[column], marker=marker, label=label)
axes[0].set_xlabel(r"Mean phonon number $\bar n$")
axes[0].set_ylabel("Taxonomy coefficient (1/gate)")
axes[0].set_title("Collective-X closure after calibration")
axes[0].grid(True, alpha=0.28)
axes[0].legend()

for condition, label, marker in [("before", "before", "o"), ("after", "after", "s")]:
    subset = comparison_long.query("condition == @condition").sort_values("n_bar")
    axes[1].plot(
        subset["n_bar"], subset["three_component_residual_fraction"],
        marker=marker, label=label,
    )
axes[1].axhline(MODEL_RESIDUAL_FRACTION_MAX, color="tab:red", linestyle="--", label="operational threshold")
axes[1].set_xlabel(r"Mean phonon number $\bar n$")
axes[1].set_ylabel(r"$\|K-K_3\|_F/\|K\|_F$")
axes[1].set_title("Three-component model adequacy")
axes[1].grid(True, alpha=0.28)
axes[1].legend()

figure.tight_layout()
diagnostic_png = ANALYSIS_DIR / "three_component_diagnostics.png"
diagnostic_pdf = ANALYSIS_DIR / "three_component_diagnostics.pdf"
figure.savefig(diagnostic_png, dpi=300, bbox_inches="tight")
figure.savefig(diagnostic_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {collective_diagnostic_path}")

,n_bar,condition,gamma_col_per_gate,taxonomy_S_IX,taxonomy_S_XI,taxonomy_C_IX_XI,collective_x_relative_spread,three_component_residual_fraction,three_component_channel_error_fraction
0,0.01,after,0.000672391,0.000561964,0.000561973,0.000536981,0.045142,0.276555,0.000742056
1,0.01,before,0.000661033,0.000550608,0.000550613,0.000525623,0.0460839,0.0378786,0.000743583
2,1,after,0.00123318,0.00112275,0.00112276,0.00109776,0.0224303,0.151767,0.000732581
3,1,before,0.00118085,0.00107043,0.00107043,0.00104544,0.0235326,0.0178812,0.000735115
4,2,after,0.0017995,0.00168909,0.00168908,0.00166409,0.0148722,0.0972455,0.000732399
5,2,before,0.00167714,0.00156672,0.00156673,0.00154173,0.0160393,0.0117722,0.000736044
6,4,after,0.00292998,0.00281956,0.00281955,0.00279456,0.00889246,0.0497417,0.000735512
7,4,before,0.00258662,0.00247619,0.00247621,0.0024512,0.0101321,0.00709229,0.000741039
8,6,after,0.00406432,0.00395387,0.0039539,0.0039289,0.00633634,0.0297719,0.000740675
9,6,before,0.00340049,0.00329007,0.00329007,0.00326507,0.0076163,0.00514314,0.000747385


<Figure size 1300x480 with 2 Axes>

Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/three_component_model_diagnostics.csv


## 8. \(|\chi|\) の対数スケール補助図

三成分だけを見て結論を固定しないため、代表的な \(\bar n\) について CPTP 射影後の \(|\chi|\) を直接可視化する。右端は較正による変化量である。

In [9]:
available_nbars = np.asarray(sorted(comparison_long["n_bar"].unique()), dtype=float)
selected_nbar = float(available_nbars[np.argmin(np.abs(available_nbars - SELECTED_NBAR_FOR_CHI))])
chi_before = channel_objects[(selected_nbar, "before")]["chi"]
chi_after = channel_objects[(selected_nbar, "after")]["chi"]
chi_delta = chi_after - chi_before

pauli_labels = list(qpt_analysis._generator_design_data()[0])
matrices = [np.abs(chi_before), np.abs(chi_after), np.abs(chi_delta)]
titles = ["Before calibration", "After calibration", r"$|\Delta\chi|$"]
vmax = max(float(np.max(matrix)) for matrix in matrices)
vmin = max(vmax * 1e-7, 1e-12)

figure, axes = plt.subplots(1, 3, figsize=(17.0, 5.2))
for axis, matrix, title in zip(axes, matrices, titles):
    image = axis.imshow(matrix, origin="lower", cmap="magma", norm=LogNorm(vmin=vmin, vmax=vmax))
    axis.set_title(title)
    axis.set_xticks(range(len(pauli_labels)), pauli_labels, rotation=90, fontsize=8)
    axis.set_yticks(range(len(pauli_labels)), pauli_labels, fontsize=8)
    axis.set_xlabel("input Pauli index")
axes[0].set_ylabel("output Pauli index")
figure.colorbar(image, ax=axes, fraction=0.025, pad=0.02, label=r"$|\chi_{P,Q}|$")
figure.suptitle(rf"Trace-normalized error $\chi$ at $\bar n={selected_nbar:g}$", y=1.02)
chi_figure_png = ANALYSIS_DIR / f"chi_before_after_nbar_{legacy_nbar_stem(selected_nbar)}.png"
chi_figure_pdf = ANALYSIS_DIR / f"chi_before_after_nbar_{legacy_nbar_stem(selected_nbar)}.pdf"
figure.savefig(chi_figure_png, dpi=300, bbox_inches="tight")
figure.savefig(chi_figure_pdf, bbox_inches="tight")
display(figure)
plt.close(figure)
print(f"Saved: {chi_figure_png}")

<Figure size 1700x520 with 4 Axes>

Saved: /Users/hirainoa/Desktop/量子/project/Moon_shot__高橋PJ/MS_gate_Hirai/results/calibration_channel_comparison/chi_before_after_nbar_4p0.png


## 9. 仮説の機械的チェックと結論候補

ここでは結論を自動的に証明するのではなく、どの主張が数値に支持され、どこが反証されたかを整理する。特に三成分 model の不適合を隠さない。

In [10]:
after_rows = comparison_long.query("condition == 'after'")
median_h_reduction = float(np.median(paired["abs_h_XX_reduction_factor"]))
median_infidelity_remaining = float(np.median(paired["infidelity_remaining_fraction"]))
max_after_residual = float(after_rows["three_component_residual_fraction"].max())
max_after_collective_spread = float(after_rows["collective_x_relative_spread"].max())

decision_rows = [
    {
        "question": "角度較正は収束したか",
        "criterion": f"全点で |h_XX| <= {HXX_TOL_RAD_PER_GATE:.2g}",
        "value": bool(paired["h_XX_converged"].all()),
        "supported": bool(paired["h_XX_converged"].all()),
    },
    {
        "question": "h_XX は強く除去されたか",
        "criterion": f"median reduction >= {STRONG_HXX_REDUCTION_FACTOR:g}",
        "value": median_h_reduction,
        "supported": median_h_reduction >= STRONG_HXX_REDUCTION_FACTOR,
    },
    {
        "question": "角度較正後の infidelity floor は残るか",
        "criterion": f"median remaining fraction >= {INFIDELITY_FLOOR_REMAINING_MIN:g}",
        "value": median_infidelity_remaining,
        "supported": median_infidelity_remaining >= INFIDELITY_FLOOR_REMAINING_MIN,
    },
    {
        "question": "三成分 model は全点で十分か",
        "criterion": f"max residual fraction <= {MODEL_RESIDUAL_FRACTION_MAX:g}",
        "value": max_after_residual,
        "supported": max_after_residual <= MODEL_RESIDUAL_FRACTION_MAX,
    },
    {
        "question": "collective-X の係数閉包は全点で良いか",
        "criterion": f"max relative spread <= {COLLECTIVE_SPREAD_RELATIVE_MAX:g}",
        "value": max_after_collective_spread,
        "supported": max_after_collective_spread <= COLLECTIVE_SPREAD_RELATIVE_MAX,
    },
]
decision_table = pd.DataFrame(decision_rows)
decision_path = ANALYSIS_DIR / "hypothesis_checks.csv"
decision_table.to_csv(decision_path, index=False)
display(decision_table)

floor_message = (
    "角度誤差を強く抑えてもチャネル infidelity の相当部分が残る。単一角度較正で除去できない誤差床、という主張の候補になる。"
    if median_h_reduction >= STRONG_HXX_REDUCTION_FACTOR
    and median_infidelity_remaining >= INFIDELITY_FLOOR_REMAINING_MIN
    else "このデータだけでは、単一角度較正後の誤差床という主張はまだ十分に支持されない。"
)
model_message = (
    "三成分 model は設定した残差基準を全点で満たす。"
    if max_after_residual <= MODEL_RESIDUAL_FRACTION_MAX
    else "三成分 model は少なくとも一部の温度で残差基準を破るため、残差の主要 H/S/C/A 成分を追加調査する必要がある。"
)

display(Markdown(
    "### 暫定結論\n\n"
    f"- {floor_message}\n"
    f"- {model_message}\n"
    "- ここで示すのは数値 QPT 内での calibration-aware comparison であり、全 MS ゲートへの普遍性や実験での因果を証明するものではない。\n"
    "- 論文の中心図にする前に、数値収束、QPT 不確かさ、固定較正値の温度間 transfer、実験 process matrix での再現を追加する。"
))

,question,criterion,value,supported
0,角度較正は収束したか,全点で |h_XX| <= 0.002,True,True
1,h_XX は強く除去されたか,median reduction >= 10,167.425,True
2,角度較正後の infidelity floor は残るか,median remaining fraction >= 0.5,0.801961,True
3,三成分 model は全点で十分か,max residual fraction <= 0.25,0.276555,False
4,collective-X の係数閉包は全点で良いか,max relative spread <= 0.25,0.045142,True


### 暫定結論

- 角度誤差を強く抑えてもチャネル infidelity の相当部分が残る。単一角度較正で除去できない誤差床、という主張の候補になる。
- 三成分 model は少なくとも一部の温度で残差基準を破るため、残差の主要 H/S/C/A 成分を追加調査する必要がある。
- ここで示すのは数値 QPT 内での calibration-aware comparison であり、全 MS ゲートへの普遍性や実験での因果を証明するものではない。
- 論文の中心図にする前に、数値収束、QPT 不確かさ、固定較正値の温度間 transfer、実験 process matrix での再現を追加する。

## 10. QEC 向けの $\gamma_{\rm col}$ 抑制パルス設計

In [18]:
import multiprocessing as mp

import laser_pulse_optimization as pulse_analysis
import phonon_xx_angle_analysis as fock_analysis

pulse_analysis = importlib.reload(pulse_analysis)
fock_analysis = importlib.reload(fock_analysis)

# ---- 編集可能な設計条件 ----
PULSE_DESIGN_NBARS = (float(SELECTED_NBAR_FOR_CHI),)
PULSE_CONTROL_POINTS = 7             # [0, a1, a2, a3, a2, a1, 0]
PULSE_ZERO_ENDPOINTS = True
PULSE_WORKER_COUNT = 8              # パルス探索と最終 QPT の worker 数
if PULSE_WORKER_COUNT < 1:
    raise ValueError("PULSE_WORKER_COUNT must be at least 1")
PULSE_OPT_MAXITER = 24
PULSE_OPT_POPSIZE = 6
PULSE_OPT_SEED = 20260901
PULSE_THERMAL_TAIL_TOL = 2e-3
PULSE_PHONON_BUFFER = 20
PULSE_GAMMA_XX_WEIGHT = 0.10         # Pauli 対角成分なので弱く抑える
PULSE_SMOOTHNESS_WEIGHT = 1e-4
PULSE_POWER_WEIGHT = 2e-5
RUN_GAMMA_COL_QPT_VALIDATION = True # False なら Fock 代理量の最適化だけ
if PULSE_WORKER_COUNT > 1 and "fork" not in mp.get_all_start_methods():
    print("fork process worker が使えないため、Fock 探索を逐次実行に切り替えます")
    PULSE_WORKER_COUNT = 1

pulse_base_params = dict(CONFIG["SIMULATION_PARAMS"])
pulse_base_amplitude = float(pulse_base_params["A"])
pulse_design_detuning = float(pulse_base_params["delta"])
configured_pulse_duration = pulse_base_params.get("t_gate_sim")
pulse_design_duration = (
    float(configured_pulse_duration)
    if configured_pulse_duration is not None
    else 2.0 * np.pi / abs(pulse_design_detuning)
)
pulse_time_points = int(pulse_base_params["time_points"])
pulse_time = np.linspace(0.0, pulse_design_duration, pulse_time_points)

calibration_match = drive_summary.loc[
    np.isclose(drive_summary["n_bar"].astype(float), PULSE_DESIGN_NBARS[0])
]
if calibration_match.empty:
    raise RuntimeError("PULSE_DESIGN_NBARS[0] に対応する角度較正結果がありません")
calibrated_rectangle_amplitude = (
    pulse_base_amplitude * float(calibration_match.iloc[-1]["A_factor"])
)
pulse_amplitude_max = 2.5 * calibrated_rectangle_amplitude
pulse_max_fock_n = max(
    fock_analysis.required_fock_cutoff(n_bar, PULSE_THERMAL_TAIL_TOL)
    for n_bar in PULSE_DESIGN_NBARS
)

def evaluate_gamma_col_waveform(amplitude):
    amplitude = np.asarray(amplitude, dtype=float)
    if amplitude.shape != pulse_time.shape:
        raise ValueError(f"amplitude must have shape {pulse_time.shape}")
    params = dict(pulse_base_params)
    params.update({
        "A": amplitude,
        "delta": pulse_design_detuning,
        "t_gate_sim": pulse_design_duration,
        "time_points": pulse_time_points,
        "use_full_order": True,
        "show_progress": False,
        "solver_max_step": pulse_design_duration / (pulse_time_points - 1),
    })
    curve = fock_analysis.calculate_fock_resolved_xx_angles(
        params,
        amplitude=amplitude,
        max_fock_n=pulse_max_fock_n,
        phonon_buffer=PULSE_PHONON_BUFFER,
        target_xx_angle_rad=np.pi / 4.0,
    )
    rows = []
    for n_bar in PULSE_DESIGN_NBARS:
        row = fock_analysis.summarize_thermal_xx_angle(
            curve, n_bar, target_xx_angle_rad=np.pi / 4.0
        )
        row["gamma_col_proxy_per_gate"] = max(
            0.0,
            row["gamma_XX_with_residual_motion_per_gate"]
            - row["gamma_XX_phase_dispersion_per_gate"],
        )
        row["pauli_compatibility_cost"] = (
            row["predicted_h_XX_rad_per_gate"] ** 2
            + row["gamma_col_proxy_per_gate"]
            + PULSE_GAMMA_XX_WEIGHT
            * row["gamma_XX_phase_dispersion_per_gate"]
        )
        rows.append(row)
    magnus = pulse_analysis.ms_magnus_metrics(
        pulse_time, amplitude, pulse_design_detuning
    )
    return {
        "curve": curve,
        "thermal_rows": pd.DataFrame(rows),
        "magnus": magnus,
    }

def gamma_col_pulse_objective(free_amplitudes, return_details=False):
    amplitude, node_times, node_amplitudes = (
        pulse_analysis.build_symmetric_laser_amplitude(
            free_amplitudes,
            pulse_time,
            PULSE_CONTROL_POINTS,
            zero_endpoints=PULSE_ZERO_ENDPOINTS,
        )
    )
    try:
        details = evaluate_gamma_col_waveform(amplitude)
    except Exception as error:
        # 個別候補の数値積分失敗で探索全体を止めない。
        if error.__class__.__name__ != "IntegratorException" or return_details:
            raise
        return 1e6 + float(np.dot(free_amplitudes, free_amplitudes))
    normalized_nodes = node_amplitudes / max(pulse_amplitude_max, 1e-15)
    smoothness = float(np.mean(np.diff(normalized_nodes, n=2) ** 2))
    relative_power = float(np.mean((amplitude / pulse_amplitude_max) ** 2))
    cost = (
        float(details["thermal_rows"]["pauli_compatibility_cost"].mean())
        + PULSE_SMOOTHNESS_WEIGHT * smoothness
        + PULSE_POWER_WEIGHT * relative_power
    )
    if return_details:
        return {
            **details,
            "cost": cost,
            "amplitude": amplitude,
            "node_times": node_times,
            "node_amplitudes": node_amplitudes,
            "smoothness_penalty": smoothness,
            "relative_power": relative_power,
        }
    return cost

pulse_free_count = (
    (PULSE_CONTROL_POINTS - 1) // 2
    if PULSE_ZERO_ENDPOINTS
    else (PULSE_CONTROL_POINTS + 1) // 2
)
pulse_node_u = np.linspace(0.0, 1.0, PULSE_CONTROL_POINTS)
pulse_initial_envelope = np.sin(np.pi * pulse_node_u) ** 2
pulse_initial_envelope /= np.sqrt(np.mean(pulse_initial_envelope**2))
pulse_initial_free = (
    calibrated_rectangle_amplitude
    * pulse_initial_envelope[1 : 1 + pulse_free_count]
)
pulse_initial_free = np.clip(pulse_initial_free, 0.0, pulse_amplitude_max)
pulse_bounds = [(0.0, pulse_amplitude_max)] * pulse_free_count

de_kwargs = dict(
    func=gamma_col_pulse_objective,
    bounds=pulse_bounds,
    x0=pulse_initial_free,
    seed=PULSE_OPT_SEED,
    maxiter=PULSE_OPT_MAXITER,
    popsize=PULSE_OPT_POPSIZE,
    polish=False,
    updating="deferred",
)
if PULSE_WORKER_COUNT == 1:
    pulse_optimization_result = scipy.optimize.differential_evolution(
        **de_kwargs, workers=1
    )
else:
    # QuTiP/ZVODE は thread-safe でないため、候補間は独立 process で並列化する。
    pulse_mp_context = mp.get_context("fork")
    with pulse_mp_context.Pool(processes=PULSE_WORKER_COUNT) as pulse_executor:
        pulse_optimization_result = scipy.optimize.differential_evolution(
            **de_kwargs, workers=pulse_executor.map
        )

pulse_optimized = gamma_col_pulse_objective(
    pulse_optimization_result.x, return_details=True
)
pulse_baseline_amplitude = np.full_like(
    pulse_time, calibrated_rectangle_amplitude
)
pulse_baseline = evaluate_gamma_col_waveform(pulse_baseline_amplitude)

pulse_design_dir = ANALYSIS_DIR / "gamma_col_pulse_design"
pulse_design_dir.mkdir(parents=True, exist_ok=True)
pulse_waveforms = pd.DataFrame({
    "time_sim": pulse_time,
    "calibrated_rectangle_A": pulse_baseline_amplitude,
    "optimized_A": pulse_optimized["amplitude"],
    "detuning": np.full_like(pulse_time, pulse_design_detuning),
})
pulse_waveforms.to_csv(pulse_design_dir / "optimized_waveform.csv", index=False)
pulse_optimized["curve"].to_csv(
    pulse_design_dir / "optimized_fock_curve.csv", index=False
)

pulse_fock_summary = pd.concat([
    pulse_baseline["thermal_rows"].assign(pulse="calibrated_rectangle"),
    pulse_optimized["thermal_rows"].assign(pulse="gamma_col_optimized"),
], ignore_index=True)
pulse_fock_summary.to_csv(
    pulse_design_dir / "fock_proxy_summary.csv", index=False
)

pulse_qpt_summary = pd.DataFrame()
if RUN_GAMMA_COL_QPT_VALIDATION:
    qpt_params = dict(pulse_base_params)
    qpt_params.update({
        "A": pulse_optimized["amplitude"],
        "delta": pulse_design_detuning,
        "t_gate_sim": pulse_design_duration,
        "time_points": pulse_time_points,
        "parallel_workers": int(PULSE_WORKER_COUNT),
        "show_progress": True,
    })
    qpt_rows = []
    for n_bar in PULSE_DESIGN_NBARS:
        baseline_channel = channel_objects[(float(n_bar), "after")]
        baseline_chi = np.asarray(baseline_channel["chi"], dtype=complex)
        baseline_offdiag = baseline_chi - np.diag(np.diag(baseline_chi))
        baseline_error_part = baseline_chi.copy()
        baseline_error_part[0, 0] = 0.0
        qpt_rows.append({
            "pulse": "calibrated_rectangle",
            "n_bar": float(n_bar),
            **baseline_channel["scalars"],
            "chi_offdiagonal_frobenius_norm": float(np.linalg.norm(baseline_offdiag)),
            "chi_offdiagonal_fraction": float(
                np.linalg.norm(baseline_offdiag)
                / max(np.linalg.norm(baseline_error_part), 1e-15)
            ),
        })
    for qpt_point in qpt_analysis.calculate_error_channel_batch(
        PULSE_DESIGN_NBARS,
        qpt_params,
        convention=CONFIG["ERROR_CHANNEL_CONVENTION"],
    ):
        channel = analyze_channel(qpt_point["chi"])
        n_bar = float(qpt_point["n_bar"])
        chi = np.asarray(channel["chi"], dtype=complex)
        chi_offdiag = chi - np.diag(np.diag(chi))
        chi_error_part = chi.copy()
        chi_error_part[0, 0] = 0.0
        qpt_rows.append({
            "pulse": "gamma_col_optimized",
            "n_bar": n_bar,
            **channel["scalars"],
            "chi_offdiagonal_frobenius_norm": float(np.linalg.norm(chi_offdiag)),
            "chi_offdiagonal_fraction": float(
                np.linalg.norm(chi_offdiag)
                / max(np.linalg.norm(chi_error_part), 1e-15)
            ),
        })
        qpt_analysis.save_qpt_point(
            pulse_design_dir / f"optimized_qpt_nbar_{legacy_nbar_stem(n_bar)}.npz",
            n_bar,
            "gamma_col_optimized",
            qpt_point["chi"],
            {**qpt_point["metadata"], "optimizer_cost": float(pulse_optimization_result.fun)},
        )
    pulse_qpt_summary = pd.DataFrame(qpt_rows)
    pulse_qpt_summary.to_csv(
        pulse_design_dir / "optimized_qpt_summary.csv", index=False
    )

pulse_design_metadata = {
    "optimizer_success": bool(pulse_optimization_result.success),
    "optimizer_message": str(pulse_optimization_result.message),
    "optimizer_cost": float(pulse_optimization_result.fun),
    "workers": int(PULSE_WORKER_COUNT),
    "control_points": int(PULSE_CONTROL_POINTS),
    "max_fock_n": int(pulse_max_fock_n),
    "design_nbars": [float(value) for value in PULSE_DESIGN_NBARS],
    "qpt_validated": bool(RUN_GAMMA_COL_QPT_VALIDATION),
}
(pulse_design_dir / "optimization_metadata.json").write_text(
    json.dumps(pulse_design_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(Markdown(
    f"**Optimization/QPT workers:** {PULSE_WORKER_COUNT}; "
    f"**max Fock n:** {pulse_max_fock_n}; "
    f"**success:** {pulse_optimization_result.success}"
))
display(pulse_fock_summary[[
    "pulse", "thermal_n_bar", "predicted_h_XX_rad_per_gate",
    "gamma_col_proxy_per_gate", "gamma_XX_phase_dispersion_per_gate",
    "pauli_compatibility_cost",
]])
if not pulse_qpt_summary.empty:
    display(pulse_qpt_summary[[
        "pulse", "n_bar", "h_XX_rad_per_gate", "gamma_XX_per_gate",
        "gamma_col_per_gate", "average_infidelity",
        "chi_offdiagonal_frobenius_norm", "chi_offdiagonal_fraction",
    ]])
print(f"Saved pulse design outputs: {pulse_design_dir}")

MS gate simulation progress: 1 n_bar values x 1 laser samples x 16 input states = 16 evolutions (16 evolutions / n_bar)
n_bar = 4.0 Simulation Finished (Dim: 73)


**Optimization/QPT workers:** 8; **max Fock n:** 27; **success:** False

,pulse,thermal_n_bar,predicted_h_XX_rad_per_gate,gamma_col_proxy_per_gate,gamma_XX_phase_dispersion_per_gate,pauli_compatibility_cost
0,calibrated_rectangle,4,5.37325e-06,0.000572995,0.00473412,0.00104641
1,gamma_col_optimized,4,-0.00184985,2.0328e-06,0.00473635,0.000479089


,pulse,n_bar,h_XX_rad_per_gate,gamma_XX_per_gate,gamma_col_per_gate,average_infidelity,chi_offdiagonal_frobenius_norm,chi_offdiagonal_fraction
0,calibrated_rectangle,4,0.00043429,0.00557183,0.00292998,0.0093656,0.00555568,0.649964
1,gamma_col_optimized,4,-0.00144722,0.00576378,0.00309087,0.00977016,0.00617348,0.673091


Saved pulse design outputs: /workspace/results/calibration_channel_comparison/gamma_col_pulse_design


## 11. 出力一覧

CSV は論文表・再解析用、NPZ は較正前後の \(\chi\) 行列、PNG/PDF は図用である。

In [19]:
manifest = {
    "analysis": "XX-angle calibration before/after channel comparison",
    "error_channel_convention": CONFIG.get("ERROR_CHANNEL_CONVENTION"),
    "calibration_protocol": "independent hXX feedback at each n_bar",
    "n_bar_values": [float(value) for value in NBAR_VALUES],
    "worker_count": int(WORKER_COUNT),
    "run_qpt": bool(RUN_QPT),
    "force_recompute": bool(FORCE_RECOMPUTE),
    "outputs": sorted(str(path.relative_to(PROJECT_ROOT)) for path in ANALYSIS_DIR.iterdir()),
    "operational_thresholds": {
        "hxx_tolerance_rad_per_gate": HXX_TOL_RAD_PER_GATE,
        "model_residual_fraction_max": MODEL_RESIDUAL_FRACTION_MAX,
        "collective_spread_relative_max": COLLECTIVE_SPREAD_RELATIVE_MAX,
        "strong_hxx_reduction_factor": STRONG_HXX_REDUCTION_FACTOR,
        "infidelity_floor_remaining_min": INFIDELITY_FLOOR_REMAINING_MIN,
    },
}
manifest_path = ANALYSIS_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame({"output": manifest["outputs"]}))
print(f"Saved: {manifest_path}")

,output
0,results/calibration_channel_comparison/channel...
1,results/calibration_channel_comparison/channel...
2,results/calibration_channel_comparison/channel...
3,results/calibration_channel_comparison/channel...
4,results/calibration_channel_comparison/chi_bef...
5,results/calibration_channel_comparison/chi_bef...
6,results/calibration_channel_comparison/chi_com...
7,results/calibration_channel_comparison/chi_com...
8,results/calibration_channel_comparison/gamma_c...
9,results/calibration_channel_comparison/hypothe...


Saved: /workspace/results/calibration_channel_comparison/manifest.json
